# LP Quality Label 对比分析\n\n
**两次标注结果对比：**\n
- **Round 1 (0204)**：`UHRS_Task_lp_relevance_labeling_0204.tsv`\n
- **Round 2 (0227)**：`UHRS_Task_lp_quality_labeling_0227-PromptRefiner_Internal100.tsv`\n\n
**分析目标：**\n
1. 各自按 ImgUrl 投票得到最终 label\n
2. 对比两轮：新增 Bad（Good→Bad）、改善 Bad（Bad→Good）\n
3. 下载各自的图片到本地目录

---\n## 公共配置 & 工具函数

In [5]:
import pandas as pd
import os
import requests
from pathlib import Path

# ================= 文件路径 =================
input_file_r1 = r"C:\Users\jinjinchen\Downloads\UHRS_Task_lp_relevance_labeling_0204.tsv"
input_file_r2 = r"D:\Data\T2I\AutoGenValidation\UHRS_Task_lp_quality_labeling_0227-PromptRefiner_Internal100.tsv"

# ================= 下载目录 =================
save_dir_r1 = r"C:\Users\jinjinchen\Downloads\label_results_0204"
save_dir_r2 = r"C:\Users\jinjinchen\Downloads\label_results_0227"

# ================= 维度列表 =================
priority_dimensions = [
    'TextLogoClarity',
    'RealisticPhysicallyPlausible',
    'BrightnessContrastColorSaturation',
    'Imageclarity',
    'SubjectClarity',
    'CompositionLayout'
]

# ================= 工具函数 =================
def vote_and_label(df):
    """
    按 ImgUrl 聚合投票：
      - 各维度 / FinalDecision：Bad 票数 >= 2 → 'Bad'，否则 'Good'
      - 返回每个 ImgUrl 一行的 DataFrame，含 FinalLabel 和各维度 label
    """
    df = df.copy()
    df['ImgUrl'] = df['ImgUrl'].astype(str).str.strip()
    df['FinalDecision'] = df['FinalDecision'].astype(str).str.strip()

    # 数字化
    for dim in priority_dimensions:
        if dim in df.columns:
            df[dim] = df[dim].astype(str).str.strip()
            df[f"{dim}_score"] = (df[dim] == 'Bad').astype(int)
        else:
            df[f"{dim}_score"] = 0
    df['Final_score'] = (df['FinalDecision'] == 'Bad').astype(int)

    agg_rules = {f"{dim}_score": 'sum' for dim in priority_dimensions}
    agg_rules['Final_score'] = 'sum'
    grouped = df.groupby('ImgUrl').agg(agg_rules).reset_index()

    # 投票 → label
    grouped['FinalLabel'] = grouped['Final_score'].apply(lambda x: 'Bad' if x >= 2 else 'Good')
    for dim in priority_dimensions:
        grouped[f'{dim}_label'] = grouped[f'{dim}_score'].apply(lambda x: 'Bad' if x >= 2 else 'Good')

    return grouped


def download_images(url_list, save_dir, label=''):
    """下载图片列表到 save_dir，跳过已存在的"""
    Path(save_dir).mkdir(parents=True, exist_ok=True)
    ok, fail = 0, 0
    for url in url_list:
        filename = url.split('/')[-1].split('?')[0] or 'image'
        save_path = os.path.join(save_dir, filename)
        if os.path.exists(save_path):
            ok += 1
            continue
        try:
            r = requests.get(url, timeout=15)
            if r.status_code == 200:
                with open(save_path, 'wb') as f:
                    f.write(r.content)
                ok += 1
            else:
                fail += 1
        except Exception:
            fail += 1
    print(f"  [{label}] 成功: {ok}, 失败: {fail}, 目录: {save_dir}")

print("配置完成")

配置完成


---\n## Round 1 (0204) - 读取 & 投票

In [6]:
df_r1_raw = pd.read_csv(input_file_r1, sep='\t', header=0)
print(f"Round 1 原始数据: {len(df_r1_raw)} 条")
print(f"列名: {df_r1_raw.columns.tolist()}")
df_r1_raw.head(3)

Round 1 原始数据: 3000 条
列名: ['JudgeID', 'HitGroupDataInt', 'HitDataInt', 'HitState', 'HitType', 'Price', 'JudgmentState', 'JudgmentDataInt', 'JudgmentDataIntName', 'JudgmentSubmitTime', 'JudgmentTypeID', 'JudgmentType', 'Consensus', 'TimeSpentOnJudgment', 'HitGroupID', 'HitID', 'lp_id', 'ImageSource', 'FinalUrl', 'ImgUrl', 'JudgmentID', 'OtherComment', 'FinalDecision', 'Imageclarity', 'BrightnessContrastColorSaturation', 'SubjectClarity', 'TextLogoClarity', 'CompositionLayout', 'RealisticPhysicallyPlausible', 'MainSubjectClarityBadOpotion']


,JudgeID,HitGroupDataInt,HitDataInt,HitState,HitType,Price,JudgmentState,JudgmentDataInt,JudgmentDataIntName,JudgmentSubmitTime,...,JudgmentID,OtherComment,FinalDecision,Imageclarity,BrightnessContrastColorSaturation,SubjectClarity,TextLogoClarity,CompositionLayout,RealisticPhysicallyPlausible,MainSubjectClarityBadOpotion
0,1628038,-1,-1,0,Normal,NaN,2,0,NaN,2/9/2026 4:17:29 PM,...,1302464073,NaN,Fair,Fair,Good,GoodOrN/A,GoodOrN/A,Good,GoodOrN/A,NaN
1,1628038,-1,-1,0,Normal,NaN,2,0,NaN,2/9/2026 4:17:51 PM,...,1302464783,NaN,Good,Good,Good,GoodOrN/A,GoodOrN/A,Good,GoodOrN/A,NaN
2,1628038,-1,-1,0,Normal,NaN,2,0,NaN,2/9/2026 4:18:13 PM,...,1302464779,NaN,Good,Good,Good,GoodOrN/A,GoodOrN/A,Good,GoodOrN/A,NaN


In [7]:
voted_r1 = vote_and_label(df_r1_raw)
bad_r1 = voted_r1[voted_r1['FinalLabel'] == 'Bad']

print(f"--- Round 1 (0204) 投票结果 ---")
print(f"唯一 URL 总数 : {len(voted_r1)}")
print(f"Bad           : {len(bad_r1)}  ({len(bad_r1)/len(voted_r1)*100:.1f}%)")
print(f"Good          : {len(voted_r1) - len(bad_r1)}")
print()
print("各维度 Bad Rate:")
for dim in priority_dimensions:
    col = f'{dim}_label'
    if col in voted_r1.columns:
        n = (voted_r1[col] == 'Bad').sum()
        print(f"  {dim}: {n}/{len(voted_r1)} ({n/len(voted_r1)*100:.1f}%)")
voted_r1.head(3)

--- Round 1 (0204) 投票结果 ---
唯一 URL 总数 : 1000
Bad           : 137  (13.7%)
Good          : 863

各维度 Bad Rate:
  TextLogoClarity: 5/1000 (0.5%)
  RealisticPhysicallyPlausible: 58/1000 (5.8%)
  BrightnessContrastColorSaturation: 0/1000 (0.0%)
  Imageclarity: 2/1000 (0.2%)
  SubjectClarity: 29/1000 (2.9%)
  CompositionLayout: 6/1000 (0.6%)


,ImgUrl,TextLogoClarity_score,RealisticPhysicallyPlausible_score,BrightnessContrastColorSaturation_score,Imageclarity_score,SubjectClarity_score,CompositionLayout_score,Final_score,FinalLabel,TextLogoClarity_label,RealisticPhysicallyPlausible_label,BrightnessContrastColorSaturation_label,Imageclarity_label,SubjectClarity_label,CompositionLayout_label
0,https://unitorchazureblob.blob.core.windows.ne...,0,1,0,0,2,1,3,Bad,Good,Good,Good,Good,Bad,Good
1,https://unitorchazureblob.blob.core.windows.ne...,0,0,0,0,0,1,1,Good,Good,Good,Good,Good,Good,Good
2,https://unitorchazureblob.blob.core.windows.ne...,0,0,0,0,0,0,0,Good,Good,Good,Good,Good,Good,Good


---\n## Round 2 (0227) - 读取 & 投票

In [8]:
df_r2_raw = pd.read_csv(input_file_r2, sep='\t', header=0)
print(f"Round 2 原始数据: {len(df_r2_raw)} 条")
print(f"列名: {df_r2_raw.columns.tolist()}")
df_r2_raw.head(3)

Round 2 原始数据: 1500 条
列名: ['JudgeID', 'HitGroupDataInt', 'HitDataInt', 'HitState', 'HitType', 'Price', 'JudgmentState', 'JudgmentDataInt', 'JudgmentDataIntName', 'JudgmentSubmitTime', 'JudgmentTypeID', 'JudgmentType', 'Consensus', 'TimeSpentOnJudgment', 'HitGroupID', 'HitID', 'lp_id', 'ImageSource', 'ImgUrl', 'JudgmentID', 'OtherComment', 'FinalDecision', 'Imageclarity', 'BrightnessContrastColorSaturation', 'SubjectClarity', 'TextLogoClarity', 'CompositionLayout', 'RealisticPhysicallyPlausible', 'MainSubjectClarityBadOpotion']


,JudgeID,HitGroupDataInt,HitDataInt,HitState,HitType,Price,JudgmentState,JudgmentDataInt,JudgmentDataIntName,JudgmentSubmitTime,...,JudgmentID,OtherComment,FinalDecision,Imageclarity,BrightnessContrastColorSaturation,SubjectClarity,TextLogoClarity,CompositionLayout,RealisticPhysicallyPlausible,MainSubjectClarityBadOpotion
0,1628038,-1,-1,0,Normal,NaN,2,0,NaN,2/27/2026 3:13:09 AM,...,1305557327,NaN,Good,Good,Good,GoodOrN/A,GoodOrN/A,Good,GoodOrN/A,NaN
1,1628038,-1,-1,0,Normal,NaN,2,0,NaN,2/27/2026 3:14:13 AM,...,1305557463,NaN,Good,Good,Good,GoodOrN/A,GoodOrN/A,Good,GoodOrN/A,NaN
2,1628038,-1,-1,0,Normal,NaN,2,0,NaN,2/27/2026 3:16:06 AM,...,1305557328,NaN,Good,Good,Good,GoodOrN/A,GoodOrN/A,Good,GoodOrN/A,NaN


In [9]:
voted_r2 = vote_and_label(df_r2_raw)
bad_r2 = voted_r2[voted_r2['FinalLabel'] == 'Bad']

print(f"--- Round 2 (0227) 投票结果 ---")
print(f"唯一 URL 总数 : {len(voted_r2)}")
print(f"Bad           : {len(bad_r2)}  ({len(bad_r2)/len(voted_r2)*100:.1f}%)")
print(f"Good          : {len(voted_r2) - len(bad_r2)}")
print()
print("各维度 Bad Rate:")
for dim in priority_dimensions:
    col = f'{dim}_label'
    if col in voted_r2.columns:
        n = (voted_r2[col] == 'Bad').sum()
        print(f"  {dim}: {n}/{len(voted_r2)} ({n/len(voted_r2)*100:.1f}%)")
voted_r2.head(3)

--- Round 2 (0227) 投票结果 ---
唯一 URL 总数 : 500
Bad           : 150  (30.0%)
Good          : 350

各维度 Bad Rate:
  TextLogoClarity: 5/500 (1.0%)
  RealisticPhysicallyPlausible: 109/500 (21.8%)
  BrightnessContrastColorSaturation: 1/500 (0.2%)
  Imageclarity: 1/500 (0.2%)
  SubjectClarity: 7/500 (1.4%)
  CompositionLayout: 1/500 (0.2%)


,ImgUrl,TextLogoClarity_score,RealisticPhysicallyPlausible_score,BrightnessContrastColorSaturation_score,Imageclarity_score,SubjectClarity_score,CompositionLayout_score,Final_score,FinalLabel,TextLogoClarity_label,RealisticPhysicallyPlausible_label,BrightnessContrastColorSaturation_label,Imageclarity_label,SubjectClarity_label,CompositionLayout_label
0,https://unitorchazureblob.blob.core.windows.ne...,0,2,0,0,0,0,2,Bad,Good,Bad,Good,Good,Good,Good
1,https://unitorchazureblob.blob.core.windows.ne...,0,0,0,0,0,1,1,Good,Good,Good,Good,Good,Good,Good
2,https://unitorchazureblob.blob.core.windows.ne...,0,1,0,0,1,2,3,Bad,Good,Good,Good,Good,Good,Bad


---\n## 对比分析：Round1 vs Round2

In [ ]:
import re

def extract_key(url):
    """从 URL 文件名中提取 InternalXXXX_PromptX 作为匹配 key（忽略日期前缀和扩展名）"""
    filename = url.split('/')[-1].split('?')[0]
    m = re.search(r'(Internal\d+_Prompt\d+)', filename, re.IGNORECASE)
    return m.group(1) if m else None

voted_r1['_key'] = voted_r1['ImgUrl'].apply(extract_key)
voted_r2['_key'] = voted_r2['ImgUrl'].apply(extract_key)

# 检查提取效果
print("Round1 key 样例:")
print(voted_r1[['ImgUrl', '_key']].head(3).to_string(index=False))
print("\nRound2 key 样例:")
print(voted_r2[['ImgUrl', '_key']].head(3).to_string(index=False))

miss_r1 = voted_r1[voted_r1['_key'].isna()]
miss_r2 = voted_r2[voted_r2['_key'].isna()]
print(f"\nRound1 key 提取失败: {len(miss_r1)} 条")
print(f"Round2 key 提取失败: {len(miss_r2)} 条")

### key 匹配对比：Round1 vs Round2

In [ ]:
# 用 _key 做 inner join
merged = pd.merge(
    voted_r1[['ImgUrl', 'FinalLabel', '_key']].rename(columns={'ImgUrl': 'ImgUrl_R1', 'FinalLabel': 'Label_R1'}),
    voted_r2[['ImgUrl', 'FinalLabel', '_key']].rename(columns={'ImgUrl': 'ImgUrl_R2', 'FinalLabel': 'Label_R2'}),
    on='_key', how='inner'
)

new_bad   = merged[(merged['Label_R1'] == 'Good') & (merged['Label_R2'] == 'Bad')]
recovered = merged[(merged['Label_R1'] == 'Bad')  & (merged['Label_R2'] == 'Good')]
stay_bad  = merged[(merged['Label_R1'] == 'Bad')  & (merged['Label_R2'] == 'Bad')]
stay_good = merged[(merged['Label_R1'] == 'Good') & (merged['Label_R2'] == 'Good')]

print(f"key 匹配成功的 URL 对数 : {len(merged)}")
print(f"  持续 Good              : {len(stay_good)}")
print(f"  持续 Bad               : {len(stay_bad)}")
print(f"  新增 Bad (Good→Bad)    : {len(new_bad)}")
print(f"  改善 (Bad→Good)        : {len(recovered)}")

only_r1_keys = set(voted_r1['_key'].dropna()) - set(voted_r2['_key'].dropna())
only_r2_keys = set(voted_r2['_key'].dropna()) - set(voted_r1['_key'].dropna())
print(f"\n  仅 Round1 独有 key: {len(only_r1_keys)}")
print(f"  仅 Round2 独有 key: {len(only_r2_keys)}")

print(f"\n=== 新增 Bad (Good→Bad)，共 {len(new_bad)} 张 ===")
for _, row in new_bad.iterrows():
    print(f"  key={row['_key']}")
    print(f"    R1: {row['ImgUrl_R1']}")
    print(f"    R2: {row['ImgUrl_R2']}")

print(f"\n=== 改善 (Bad→Good)，共 {len(recovered)} 张 ===")
for _, row in recovered.iterrows():
    print(f"  key={row['_key']}")
    print(f"    R1: {row['ImgUrl_R1']}")
    print(f"    R2: {row['ImgUrl_R2']}")

---\n## 下载图片到本地

In [ ]:
# ================= 下载 Round1 所有 Bad 图片 =================
print("下载 Round1 (0204) Bad 图片...")
download_images(bad_r1['ImgUrl'].tolist(), os.path.join(save_dir_r1, 'bad'), label='R1-Bad')

# ================= 下载 Round2 所有 Bad 图片 =================
print("下载 Round2 (0227) Bad 图片...")
download_images(bad_r2['ImgUrl'].tolist(), os.path.join(save_dir_r2, 'bad'), label='R2-Bad')

# ================= 下载新增 Bad（Good→Bad），用 R2 的 URL =================
print("下载新增 Bad (Good→Bad)...")
download_images(new_bad['ImgUrl_R2'].tolist(), os.path.join(save_dir_r2, 'new_bad_good_to_bad'), label='New-Bad')

# ================= 下载改善（Bad→Good），用 R2 的 URL =================
print("下载改善图片 (Bad→Good)...")
download_images(recovered['ImgUrl_R2'].tolist(), os.path.join(save_dir_r2, 'recovered_bad_to_good'), label='Recovered')

print("\n下载完成！目录结构：")
print(f"  {save_dir_r1}/bad/                   ← Round1 Bad 图片")
print(f"  {save_dir_r2}/bad/                   ← Round2 Bad 图片")
print(f"  {save_dir_r2}/new_bad_good_to_bad/   ← 新增 Bad (Good→Bad)")
print(f"  {save_dir_r2}/recovered_bad_to_good/ ← 改善 (Bad→Good)")